# DriftWatch: EDA, Training, MLflow, and Model v1

This notebook is intentionally thin: reusable production logic lives in `platform/app/ml`, while the notebook focuses on explanation, EDA, and readable review outputs.

## A. Setup

The dataset expected by this branch is `platform/data/raw/bank-additional-full.csv`. The training code reads config from environment-backed settings and keeps the operating threshold separate from the model artifact.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
PLATFORM_ROOT = PROJECT_ROOT / "platform"
sys.path.insert(0, str(PLATFORM_ROOT))

from app.config import get_ml_settings
from app.ml.data import clean_bank_marketing_data, load_bank_marketing_data, make_train_test_split, split_features_target
from app.ml.eda import (
    categorical_cardinality_with_target_percentages,
    duplicate_summary,
    get_numeric_categorical_columns,
    missing_values_summary,
    numeric_distribution_summary,
    summarize_columns,
)
from app.ml.train import run_training_pipeline

settings = get_ml_settings()
DATA_PATH = PROJECT_ROOT / settings.data_path
sns.set_theme(style="whitegrid")
DATA_PATH


## B. Load Dataset

We load the raw UCI Bank Marketing file before any cleaning so the EDA can explicitly show leakage and sentinel issues.

In [ ]:
raw_df = load_bank_marketing_data(DATA_PATH)
display(raw_df.shape)
display(raw_df.head())


## C. EDA

The current `bank-full.csv` is imbalanced, includes `duration` leakage, and uses `pdays == -1` as a sentinel for clients not previously contacted.

In [ ]:
display(summarize_columns(raw_df))
numeric_cols, categorical_cols = get_numeric_categorical_columns(raw_df, target_col="y")
print(f"numeric columns: {len(numeric_cols)}")
print(f"categorical columns: {len(categorical_cols)}")
display(pd.Series(categorical_cols, name="categorical_columns"))
display(pd.Series(numeric_cols, name="numeric_columns"))


Observation: high-cardinality categorical columns can create wide one-hot encodings, while macroeconomic numeric features such as employment or interest-rate indicators may be drift-sensitive.

In [ ]:
display(raw_df["y"].value_counts().rename("target_count"))
display(raw_df["y"].value_counts(normalize=True).rename("target_percent") * 100)
display(missing_values_summary(raw_df))
display(duplicate_summary(raw_df))


Observation: the target class is expected to be imbalanced, so recall and thresholding matter more than plain accuracy.

In [ ]:
display(categorical_cardinality_with_target_percentages(raw_df).head(80))
display(numeric_distribution_summary(raw_df))


In [ ]:
plot_numeric = numeric_cols[:8]
raw_df[plot_numeric].hist(figsize=(14, 10), bins=30)
plt.suptitle("Numeric feature distributions")
plt.tight_layout()
plt.show()

for column in plot_numeric:
    plt.figure(figsize=(7, 2.8))
    sns.boxplot(data=raw_df, x=column)
    plt.title(f"Outlier check: {column}")
    plt.tight_layout()
    plt.show()


Observation: skewed numeric features and outliers are expected in campaign and economic data. Tree models can handle this naturally; LR benefits from scaling.

In [ ]:
for column in categorical_cols[:8]:
    plt.figure(figsize=(10, 3.5))
    order = raw_df[column].value_counts().head(12).index
    sns.countplot(data=raw_df, y=column, order=order)
    plt.title(f"Top categories: {column}")
    plt.tight_layout()
    plt.show()


In [ ]:
corr_df = raw_df.copy()
corr_df["y_numeric"] = corr_df["y"].map({"yes": 1, "no": 0})
corr_cols = [*numeric_cols, "y_numeric"]
plt.figure(figsize=(10, 8))
sns.heatmap(corr_df[corr_cols].corr(numeric_only=True), cmap="vlag", center=0)
plt.title("Numeric correlation heatmap")
plt.tight_layout()
plt.show()


Observation: `duration` is likely correlated with the target because it is measured after the call. We drop it before modeling to avoid leakage.

## D. Cleaning

Cleaning is delegated to `clean_bank_marketing_data` so the notebook and CLI share the same rules.

In [ ]:
before_columns = set(raw_df.columns)
clean_df = clean_bank_marketing_data(raw_df)
after_columns = set(clean_df.columns)

print("removed columns:", sorted(before_columns - after_columns))
print("duration removed:", "duration" not in clean_df.columns)
print("unknown remains:", (clean_df == "unknown").any().any())
print("pdays_was_minus_one exists:", "pdays_was_minus_one" in clean_df.columns)
display(clean_df.head())


## E. Split

The required split is 70/30 with stratification and `random_state=42`.

In [ ]:
X, y = split_features_target(clean_df)
X_train, X_test, y_train, y_test = make_train_test_split(X, y, test_size=0.30, random_state=42)
display(pd.DataFrame({
    "split": ["train", "test"],
    "rows": [len(y_train), len(y_test)],
    "positive_rate": [y_train.mean(), y_test.mean()],
}))


## F-I. Training, Tuning, Threshold, Evaluation

The CLI-style pipeline trains baseline models, tunes LR/RF/HGB with small searches, logs runs to MLflow, chooses the best tuned model by CV AUC among models satisfying recall >= 0.75, tunes the threshold separately, and evaluates on the test split.

In [ ]:
summary = run_training_pipeline(
    data_path=DATA_PATH,
    artifact_root=PROJECT_ROOT / settings.artifact_dir,
    mlflow_tracking_uri=settings.mlflow_tracking_uri,
    mlflow_experiment_name=settings.mlflow_experiment_name,
    registered_model_name=settings.mlflow_registered_model_name,
    random_state=settings.random_state,
    test_size=settings.test_size,
    min_recall=settings.min_recall,
    model_version_label=settings.model_version_label,
)
display(summary)


Why threshold is separate: `model.pkl` returns probabilities. `threshold.json` controls the business operating point and can be changed or audited without repickling the pipeline.

## J-K. Artifacts and MLflow Registration

The training call creates `platform/artifacts/model_v1/model.pkl`, `schema.json`, `card.md`, `metrics.json`, `threshold.json`, and `run_metadata.json`. It also logs all runs to the `DriftWatch Bank Marketing` experiment and registers only the selected best model as `driftwatch-bank-marketing`.

Promotion to Production should happen later through the platform promotion gate, not directly from this notebook.